# Estimación experimental de EPH-Gran La Plata 2007T3 vía aglomerados comparables

**Rama `experimental/estimacion-eph-2007t3` -- no mergear a `main` sin autorización explícita.**
Esto es un experimento para medir sensibilidad, no un cambio a producción por default.

## Contexto (por qué existe este notebook)

INDEC no relevó el aglomerado Gran La Plata en el 3° trimestre de 2007 -- confirmado contra la
fuente primaria oficial (`indicadores_eph_3trim07.pdf`, informe de prensa del propio 3° trimestre
de 2007): *"Durante el 3º trimestre de 2007, los Aglomerados Mar del Plata-Batán, Bahía
Blanca-Cerri y Gran La Plata no fueron relevados por causas de orden administrativo, mientras que
los casos correspondientes al Aglomerado Gran Buenos Aires no fueron relevados por paro del
personal de la EPH."* No es un accidente de extracción de este repo (`data/socioeconomia/eph_cache/2007/`
nunca tuvo ni intentó tener una carpeta `trim3`).

**Investigación previa (misma sesión) de una fuente alternativa real** -- descartada, no
encontrada: Dirección Provincial de Estadística de Buenos Aires (es la misma institución que iba
a corelevar, no salió a medir por su cuenta), EIL (releva establecimientos, no hogares -- universo
incompatible), EIMTM Municipalidad de La Plata (sí es de hogares, pero su único operativo
documentado es de abril de 2008, ~8 meses después del hueco, y solo mide variables laborales, no
las 5 variables de hogar que también usa este repo), la propia imputación de INDEC para el
agregado de 31 aglomerados (ajustó el dato de GBA con la variación del resto -- una estimación,
no una medición, y sobre GBA no sobre Gran La Plata), CEDLAS-UNLP (no releva, usa la EPH). **No
hay ninguna fuente que empalme en tiempo, geografía y metodología simultáneamente** -- a
diferencia de FACPCE/IPC. Por eso este notebook explora una estimación matemática, con todos sus
supuestos y limitaciones explícitos, en vez de dejarlo sin intentar.

**Regla dura de este experimento**: nunca reemplaza el dato real por el estimado en ningún archivo
existente. Todo vive en un archivo nuevo (`data/tfi_data/eph_estimacion_2007t3_experimental.csv`,
columna `estimado_no_medido=True`) y en paneles derivados marcados como experimentales, separados
de `data/tfi_data/panel/t-1/`.

## 1. Aglomerados de referencia

Verificado contra `EPHcont_3trim07.pdf` (Informe de Prensa "Mercado de trabajo", INDEC, sección
1.4 "Principales indicadores por áreas geográficas") -- lista completa de los 27 aglomerados
efectivamente relevados en 3T2007, agrupados por región. Dentro de la región **Pampeana**
(la misma de Gran La Plata) fueron relevados: Concordia, **Gran Córdoba**, **Gran Rosario**, Gran
Paraná, Gran Santa Fe, Río Cuarto, Santa Rosa-Toay y San Nicolás-Villa Constitución.

**Grupo de referencia elegido: Gran Córdoba (`AGLOMERADO=13`) + Gran Rosario (`AGLOMERADO=4`)**
-- son los dos únicos aglomerados Pampeana relevados que además comparten el tier de tamaño de
Gran La Plata (500.000+ habitantes; el resto de la lista Pampeana son aglomerados chicos,
menos comparables en estructura de mercado laboral urbano). Los códigos de aglomerado se
verificaron contra el ranking poblacional real de la propia base (no de memoria): el orden de
tamaños de los 32 códigos presentes coincide exactamente con las ciudades conocidas.

In [1]:
import sys
general_path = "/workspaces/analisis-politica-economia/"
sys.path.insert(0, f"{general_path}src")

import zipfile
import tempfile
from datetime import date
from pathlib import Path
import copy

import pandas as pd
import numpy as np
from dbfread import DBF

from socioeconomia.eph_client import agregados_gran_la_plata
from ml_models.cargar_series_economicas import cargar_registro, FilaRegistroVariable
from ml_models.construir_panel_ventanas import _cargar_series_mensuales, _cargar_ventanas
from ml_models.construir_elecciones_resumen import cargar_elecciones
from ml_models.construir_panel_trimestral import construir_panel_trimestral, _escribir_csv, COLUMNAS_ELECCION_PANEL, _variables_con_datos
from ml_models.panel_k import construir_panel_k, columnas_candidatas_k
from ml_models.lasso import construir_Xy_final, lasso_loocv_manual, baseline_trivial_loocv, ajustar_final, encontrar_redundantes, elegir_representante

CACHE_EPH = f"{general_path}data/socioeconomia/eph_cache"
REGISTRO_VARIABLES_PATH = f"{general_path}data/tfi_data/registro_variables.csv"
SERIES_ECONOMICAS_MENSUALES_PATH = f"{general_path}data/tfi_data/series_economicas_mensuales.csv"
VENTANAS_PATH = f"{general_path}data/tfi_data/ventanas.csv"
ELECCIONES_RESUMEN_PATH = f"{general_path}data/tfi_data/elecciones.csv"

AGLOMERADOS = {2: "gran_la_plata", 13: "gran_cordoba", 4: "gran_rosario"}
VARIABLES_EPH = (
    "tasa_informalidad", "pct_sin_cobertura_salud", "hacinamiento_medio",
    "pct_hogares_ayuda_social_gobierno", "pct_hogares_prestamo_bancario",
    "pct_hogares_vendio_pertenencias",
)


def _leer_base_zip(zip_path: Path, prefijo: str) -> pd.DataFrame:
    """Igual que `EphClient.leer_base_historica` pero sin pasar por `unar`
    (no disponible en este entorno de ejecución) -- 2006-2008 son .zip
    planos con el DBF directo adentro, `dbfread` los lee tras un
    `extractall` simple."""
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(tmp)
        candidato = [p for p in Path(tmp).iterdir() if p.name.lower().startswith(prefijo) and p.suffix.lower() == ".dbf"][0]
        tabla = DBF(str(candidato), encoding="latin-1", char_decode_errors="ignore")
        return pd.DataFrame(iter(tabla))


def cargar_trimestre(anio: int, trimestre: int) -> dict[int, dict]:
    yy = anio % 100
    zip_path = Path(CACHE_EPH) / str(anio) / f"trim{trimestre}" / f"t{trimestre}{yy:02d}_dbf.zip"
    ind = _leer_base_zip(zip_path, "ind_")
    hog = _leer_base_zip(zip_path, "hog_")
    return {cod: agregados_gran_la_plata(ind, hog, aglomerado=cod) for cod in AGLOMERADOS}, ind


PERIODOS = [(2006, t) for t in (1, 2, 3, 4)] + [(2007, t) for t in (1, 2, 4)] + [(2008, t) for t in (1, 2, 3, 4)]

filas = []
individual_por_periodo = {}
for anio, trimestre in PERIODOS:
    por_aglo, ind_completo = cargar_trimestre(anio, trimestre)
    individual_por_periodo[(anio, trimestre)] = ind_completo
    for cod, nombre in AGLOMERADOS.items():
        fila = {"anio": anio, "trimestre": trimestre, "aglomerado": cod, "nombre": nombre}
        for v in VARIABLES_EPH:
            fila[v] = por_aglo[cod][v]
        filas.append(fila)

serie_3_aglomerados = pd.DataFrame(filas)
print(f"{len(serie_3_aglomerados)} filas ({len(PERIODOS)} trimestres x {len(AGLOMERADOS)} aglomerados)")
serie_3_aglomerados

33 filas (11 trimestres x 3 aglomerados)


,anio,trimestre,aglomerado,nombre,tasa_informalidad,pct_sin_cobertura_salud,hacinamiento_medio,pct_hogares_ayuda_social_gobierno,pct_hogares_prestamo_bancario,pct_hogares_vendio_pertenencias
0,2006,1,2,gran_la_plata,0.364108,0.322175,1.105666,0.010902,0.155930,0.090680
1,2006,1,13,gran_cordoba,0.484577,0.426332,1.286251,0.179575,0.070510,0.083504
2,2006,1,4,gran_rosario,0.423896,0.315131,1.256407,0.056382,0.035800,0.039554
3,2006,2,2,gran_la_plata,0.378322,0.314132,1.081883,0.001294,0.116368,0.091228
4,2006,2,13,gran_cordoba,0.441511,0.407847,1.324153,0.174072,0.070187,0.081696
5,2006,2,4,gran_rosario,0.370143,0.273754,1.247843,0.038887,0.029484,0.045532
6,2006,3,2,gran_la_plata,0.367712,0.297801,1.065125,0.013698,0.121548,0.054984
7,2006,3,13,gran_cordoba,0.445893,0.383072,1.309334,0.152522,0.056384,0.053059
8,2006,3,4,gran_rosario,0.404443,0.262141,1.213315,0.056194,0.067559,0.050597
9,2006,4,2,gran_la_plata,0.369428,0.275983,1.068184,0.028693,0.102349,0.048976


## 2. Limitación explícita: estructura de empleo distinta (empleo estatal)

Gran La Plata tiene una proporción de empleo estatal excepcionalmente alta frente a Córdoba y
Rosario (economías con más peso industrial/comercial) -- comparten región y tier de tamaño con
GLP, pero **no necesariamente estructura de empleo**. Esto es particularmente relevante para
`tasa_informalidad`: el empleo estatal es formal por definición en la forma en que la EPH mide
informalidad (`PP07H`, "le descuentan jubilación", solo se pregunta a asalariados) -- una ciudad
con más empleo público tiene, mecánicamente, menos informalidad medida, sin que eso refleje un
proceso económico transferible desde Córdoba/Rosario.

Verificado acá con datos reales de la propia EPH (no solo citado): `PP04A` (1=estatal) sobre la
población ocupada, ponderada por `PONDERA`, en los mismos trimestres ya cargados.

In [2]:
filas_estatal = []
for (anio, trimestre), ind in individual_por_periodo.items():
    ind = ind.copy()
    ind["ESTADO"] = pd.to_numeric(ind["ESTADO"], errors="coerce")
    ind["PP04A"] = pd.to_numeric(ind["PP04A"], errors="coerce")
    ind["PONDERA"] = pd.to_numeric(ind["PONDERA"], errors="coerce")
    for cod, nombre in AGLOMERADOS.items():
        sub = ind[(ind["AGLOMERADO"] == cod) & (ind["ESTADO"] == 1)]
        total = sub["PONDERA"].sum()
        estatal = sub.loc[sub["PP04A"] == 1, "PONDERA"].sum()
        filas_estatal.append({
            "anio": anio, "trimestre": trimestre, "nombre": nombre,
            "pct_empleo_estatal_sobre_ocupados": 100 * estatal / total if total else None,
        })

empleo_estatal = pd.DataFrame(filas_estatal)
resumen_estatal = empleo_estatal.groupby("nombre")["pct_empleo_estatal_sobre_ocupados"].agg(["mean", "min", "max"])
print(resumen_estatal)

                    mean        min        max
nombre                                        
gran_cordoba   12.947313   9.558275  15.203195
gran_la_plata  32.086284  28.939195  35.611565
gran_rosario   11.338762   9.319326  14.211268


**Confirmado con datos reales**: Gran La Plata tiene ~29-36% de empleo estatal sobre ocupados en
estos 11 trimestres, contra ~9-15% en Córdoba y Rosario -- 2 a 3 veces más, consistente con la
cifra de referencia (~38%, IELAP) citada para este experimento. Esta es una limitación estructural
del grupo de referencia, no un detalle menor: cualquier estimación de `tasa_informalidad` para
Gran La Plata basada en el movimiento de Córdoba/Rosario hereda este sesgo.

## 3. Validación del supuesto de paralelismo (con datos reales, antes de estimar nada)

Para los pares de trimestres consecutivos **dentro de 2006** (T1→T2, T2→T3, T3→T4) y **dentro de
2008** (T1→T2, T2→T3, T3→T4) -- 6 pares en total, los únicos donde Gran La Plata, Córdoba y
Rosario tienen los 3 dato real -- se compara el movimiento real de Gran La Plata contra el
promedio del movimiento de Córdoba+Rosario, para las 6 variables. Si el grupo de referencia
predijera bien el movimiento de Gran La Plata, la correlación sería alta y el error de predicción
bajo (comparado con la variabilidad real de Gran La Plata). **Se reporta el resultado tal cual
da, sea bueno o malo.**

In [3]:
PARES_VALIDACION = [(2006,1,2006,2), (2006,2,2006,3), (2006,3,2006,4),
                     (2008,1,2008,2), (2008,2,2008,3), (2008,3,2008,4)]

def valor(anio, trim, cod):
    return serie_3_aglomerados[(serie_3_aglomerados.anio==anio)&(serie_3_aglomerados.trimestre==trim)&(serie_3_aglomerados.aglomerado==cod)].iloc[0]

filas_val = []
for a1,t1,a2,t2 in PARES_VALIDACION:
    glp1, glp2 = valor(a1,t1,2), valor(a2,t2,2)
    cor1, cor2 = valor(a1,t1,13), valor(a2,t2,13)
    ros1, ros2 = valor(a1,t1,4), valor(a2,t2,4)
    for v in VARIABLES_EPH:
        delta_glp = glp2[v] - glp1[v]
        delta_ref = ((cor2[v]-cor1[v]) + (ros2[v]-ros1[v])) / 2
        filas_val.append({
            "par": f"{a1}T{t1}->{a2}T{t2}", "variable": v,
            "delta_glp_real": delta_glp, "delta_ref_promedio": delta_ref,
            "error": delta_ref - delta_glp,
        })

validacion_detalle = pd.DataFrame(filas_val)

resumen_validacion = validacion_detalle.groupby("variable").apply(lambda g: pd.Series({
    "n_pares": len(g),
    "corr_delta_ref_vs_delta_glp_real": g["delta_glp_real"].corr(g["delta_ref_promedio"]),
    "mae_estimacion": g["error"].abs().mean(),
    "rmse_estimacion": (g["error"]**2).mean()**0.5,
    "sd_delta_glp_real": g["delta_glp_real"].std(),
})).reset_index()
resumen_validacion["rmse_peor_que_prediccion_trivial_cero"] = resumen_validacion["rmse_estimacion"] > resumen_validacion["sd_delta_glp_real"]
resumen_validacion

,variable,n_pares,corr_delta_ref_vs_delta_glp_real,mae_estimacion,rmse_estimacion,sd_delta_glp_real,rmse_peor_que_prediccion_trivial_cero
0,hacinamiento_medio,6.0,0.401856,0.023679,0.025480,0.018424,True
1,pct_hogares_ayuda_social_gobierno,6.0,0.051001,0.012052,0.015566,0.010408,True
2,pct_hogares_prestamo_bancario,6.0,0.129514,0.024496,0.030380,0.017610,True
3,pct_hogares_vendio_pertenencias,6.0,0.534033,0.014669,0.016810,0.018753,False
4,pct_sin_cobertura_salud,6.0,0.021502,0.023826,0.026375,0.019596,True
5,tasa_informalidad,6.0,-0.559815,0.026633,0.032151,0.013321,True


**Resultado, reportado sin filtrar**: para **las 6 variables**, el RMSE de usar el movimiento de
Córdoba+Rosario como predictor es **mayor** que el desvío estándar del propio movimiento real de
Gran La Plata -- es decir, en estos 6 pares verificables, **predecir "sin cambio" (delta=0) es
mejor que usar la variación del grupo de referencia**, para cada una de las 6 variables. La
correlación es baja o directamente negativa: `tasa_informalidad` da **-0,56** (peor que nada:
cuando el grupo de referencia sube, Gran La Plata tiende a bajar en esta muestra chica),
consistente con la limitación de estructura de empleo documentada arriba.
`pct_hogares_vendio_pertenencias` (0,53) y `hacinamiento_medio` (0,40) son las menos malas, el
resto está cerca de cero o es negativo.

**Esto es una señal real de que la estimación de 2007T3 no es confiable, con N chico (6 pares) --
se documenta y se sigue adelante con el experimento tal como se pidió, no se usa como motivo para
no generarlo. El lector de este notebook tiene que ver este resultado antes de mirar cualquier
`estimado_desde_t2`/`estimado_desde_t4` de las celdas de abajo.**

## 4. Fórmula exacta

Ancla: valores reales de Gran La Plata en 2007T2 y 2007T4. Variación de referencia: movimiento
real de Córdoba+Rosario (promedio simple de los dos aglomerados) entre 2007T2 y 2007T4.

```
delta_ref = mean(Cordoba_T4 - Cordoba_T2, Rosario_T4 - Rosario_T2)
estimado_desde_t2 = GLP_T2 + delta_ref / 2
estimado_desde_t4 = GLP_T4 - delta_ref / 2
```

**Por qué NO se promedian las dos proyecciones** (hallazgo previo a escribir código, verificado
por álgebra): `(estimado_desde_t2 + estimado_desde_t4) / 2 = (GLP_T2 + GLP_T4) / 2` -- el término
`delta_ref` se cancela exactamente, sin importar su valor. Promediar ambas proyecciones es
matemáticamente idéntico a interpolar la propia serie de Gran La Plata sola, exactamente lo que
este experimento existe para evitar (el principio ya establecido: no imputar sobre la propia
serie). Por eso las dos proyecciones se guardan y se usan **por separado**, nunca combinadas.

In [4]:
glp_t2, glp_t4 = valor(2007,2,2), valor(2007,4,2)
cor_t2, cor_t4 = valor(2007,2,13), valor(2007,4,13)
ros_t2, ros_t4 = valor(2007,2,4), valor(2007,4,4)

filas_estim = []
for v in VARIABLES_EPH:
    delta_ref = ((cor_t4[v]-cor_t2[v]) + (ros_t4[v]-ros_t2[v])) / 2
    est_t2 = glp_t2[v] + delta_ref/2
    est_t4 = glp_t4[v] - delta_ref/2
    filas_estim.append({
        "variable": v,
        "glp_t2_real": glp_t2[v], "glp_t4_real": glp_t4[v],
        "cordoba_t2": cor_t2[v], "cordoba_t4": cor_t4[v],
        "rosario_t2": ros_t2[v], "rosario_t4": ros_t4[v],
        "delta_ref_t2_t4": delta_ref,
        "estimado_desde_t2": est_t2, "estimado_desde_t4": est_t4,
        "divergencia_entre_proyecciones": est_t2 - est_t4,
    })

estimacion_2007t3 = pd.DataFrame(filas_estim)
estimacion_2007t3

,variable,glp_t2_real,glp_t4_real,cordoba_t2,cordoba_t4,rosario_t2,rosario_t4,delta_ref_t2_t4,estimado_desde_t2,estimado_desde_t4,divergencia_entre_proyecciones
0,tasa_informalidad,0.376785,0.306393,0.404325,0.402522,0.372088,0.402133,0.014121,0.383846,0.299333,0.084513
1,pct_sin_cobertura_salud,0.305635,0.256557,0.349402,0.350717,0.268076,0.267348,0.000294,0.305782,0.256411,0.049371
2,hacinamiento_medio,1.065990,1.053753,1.265569,1.308848,1.221850,1.269504,0.045467,1.088723,1.031020,0.057703
3,pct_hogares_ayuda_social_gobierno,0.037173,0.020367,0.149234,0.124059,0.063444,0.076760,-0.005930,0.034208,0.023332,0.010876
4,pct_hogares_prestamo_bancario,0.091012,0.114024,0.065272,0.069379,0.050554,0.044934,-0.000756,0.090634,0.114402,-0.023768
5,pct_hogares_vendio_pertenencias,0.033790,0.041885,0.043174,0.044892,0.030112,0.031091,0.001349,0.034464,0.041210,-0.006746


## 5. Guardar el archivo experimental

`data/tfi_data/eph_estimacion_2007t3_experimental.csv` -- archivo nuevo, separado, nunca
sobrescribe `eph_gran_la_plata.csv`. `estimado_no_medido=True` en toda fila (las 6, una por
variable).

In [5]:
salida = estimacion_2007t3.copy()
salida.insert(0, "anio", 2007)
salida.insert(1, "trimestre", 3)
salida["aglomerados_referencia"] = "gran_cordoba(13)+gran_rosario(4)"
salida["metodo"] = "ancla_real_glp_t2_t4 + variacion_real_referencia_t2_t4 (proyecciones sin promediar)"
salida["estimado_no_medido"] = True

destino = f"{general_path}data/tfi_data/eph_estimacion_2007t3_experimental.csv"
salida.to_csv(destino, index=False)
print(f"escrito: {destino}")
salida

escrito: /workspaces/analisis-politica-economia/data/tfi_data/eph_estimacion_2007t3_experimental.csv


,anio,trimestre,variable,glp_t2_real,glp_t4_real,cordoba_t2,cordoba_t4,rosario_t2,rosario_t4,delta_ref_t2_t4,estimado_desde_t2,estimado_desde_t4,divergencia_entre_proyecciones,aglomerados_referencia,metodo,estimado_no_medido
0,2007,3,tasa_informalidad,0.376785,0.306393,0.404325,0.402522,0.372088,0.402133,0.014121,0.383846,0.299333,0.084513,gran_cordoba(13)+gran_rosario(4),ancla_real_glp_t2_t4 + variacion_real_referenc...,True
1,2007,3,pct_sin_cobertura_salud,0.305635,0.256557,0.349402,0.350717,0.268076,0.267348,0.000294,0.305782,0.256411,0.049371,gran_cordoba(13)+gran_rosario(4),ancla_real_glp_t2_t4 + variacion_real_referenc...,True
2,2007,3,hacinamiento_medio,1.065990,1.053753,1.265569,1.308848,1.221850,1.269504,0.045467,1.088723,1.031020,0.057703,gran_cordoba(13)+gran_rosario(4),ancla_real_glp_t2_t4 + variacion_real_referenc...,True
3,2007,3,pct_hogares_ayuda_social_gobierno,0.037173,0.020367,0.149234,0.124059,0.063444,0.076760,-0.005930,0.034208,0.023332,0.010876,gran_cordoba(13)+gran_rosario(4),ancla_real_glp_t2_t4 + variacion_real_referenc...,True
4,2007,3,pct_hogares_prestamo_bancario,0.091012,0.114024,0.065272,0.069379,0.050554,0.044934,-0.000756,0.090634,0.114402,-0.023768,gran_cordoba(13)+gran_rosario(4),ancla_real_glp_t2_t4 + variacion_real_referenc...,True
5,2007,3,pct_hogares_vendio_pertenencias,0.033790,0.041885,0.043174,0.044892,0.030112,0.031091,0.001349,0.034464,0.041210,-0.006746,gran_cordoba(13)+gran_rosario(4),ancla_real_glp_t2_t4 + variacion_real_referenc...,True


## 6. Los 3 escenarios de `panel_trimestral` (en memoria, nunca sobre los CSV reales)

`construir_panel_trimestral` acepta `series_mensuales` como diccionario en memoria (no lee de
disco) -- se parte de la serie mensual REAL ya cacheada
(`data/tfi_data/series_economicas_mensuales.csv`, sin red), y para los 2 escenarios "con
estimación" se sobreescribe, **solo en la copia en memoria de las 6 variables EPH**, los 3 meses
de 2007-07/08/09 con el valor estimado correspondiente -- exactamente como
`_homogeneizar_mensual` (periodicidad trimestral) ya repite cualquier trimestre real dentro de
sus 3 meses. Los 3 escenarios se escriben a
`data/tfi_data/panel/experimental_eph2007t3/<escenario>/panel_trimestral_<nivel>.csv` -- nunca a
`data/tfi_data/panel/t-1/` (el real, intacto).

In [6]:
NIVELES = ("municipal", "provincial", "nacional")
MESES_2007T3 = [date(2007, 7, 1), date(2007, 8, 1), date(2007, 9, 1)]

registro = cargar_registro(REGISTRO_VARIABLES_PATH)
ventanas = _cargar_ventanas(VENTANAS_PATH)
elecciones_por_anio_nivel = cargar_elecciones(ELECCIONES_RESUMEN_PATH)
series_mensuales_real = _cargar_series_mensuales(SERIES_ECONOMICAS_MENSUALES_PATH, registro)

# confirma el hueco real antes de tocar nada
for v in VARIABLES_EPH:
    print(v, [series_mensuales_real[v][m] for m in MESES_2007T3])

tasa_informalidad [None, None, None]
pct_sin_cobertura_salud [None, None, None]
hacinamiento_medio [None, None, None]
pct_hogares_ayuda_social_gobierno [None, None, None]
pct_hogares_prestamo_bancario [None, None, None]
pct_hogares_vendio_pertenencias [None, None, None]


In [7]:
def series_con_estimacion(columna_estimado: str) -> dict:
    serie = copy.deepcopy(series_mensuales_real)
    fila_por_variable = estimacion_2007t3.set_index("variable")[columna_estimado]
    for v in VARIABLES_EPH:
        for mes in MESES_2007T3:
            serie[v][mes] = float(fila_por_variable[v])
    return serie

ESCENARIOS = {
    "sin_estimar": series_mensuales_real,
    "con_estimado_desde_t2": series_con_estimacion("estimado_desde_t2"),
    "con_estimado_desde_t4": series_con_estimacion("estimado_desde_t4"),
}

DESTINO_BASE = f"{general_path}data/tfi_data/panel/experimental_eph2007t3"
variables_reg = _variables_con_datos(registro, series_mensuales_real)
columnas_panel = [
    "id_transicion", "nivel", "anio_t", "anio_t_menos_1", "orden", "tipo_fila",
    "fecha_inicio", "fecha_fin", "n_meses",
] + COLUMNAS_ELECCION_PANEL + sorted(var.id_variable for var in variables_reg)

for escenario, series in ESCENARIOS.items():
    for nivel in NIVELES:
        filas_panel = construir_panel_trimestral(ventanas, registro, series, elecciones_por_anio_nivel, nivel)
        destino = _escribir_csv(f"{DESTINO_BASE}/{escenario}/panel_trimestral_{nivel}.csv", filas_panel, columnas_panel)
    print(f"{escenario}: escrito en {DESTINO_BASE}/{escenario}/")

sin_estimar: escrito en /workspaces/analisis-politica-economia/data/tfi_data/panel/experimental_eph2007t3/sin_estimar/
con_estimado_desde_t2: escrito en /workspaces/analisis-politica-economia/data/tfi_data/panel/experimental_eph2007t3/con_estimado_desde_t2/
con_estimado_desde_t4: escrito en /workspaces/analisis-politica-economia/data/tfi_data/panel/experimental_eph2007t3/con_estimado_desde_t4/


In [8]:
# confirmación explícita de que el escenario "sin_estimar" no cambió nada respecto del real
import filecmp
for nivel in NIVELES:
    igual = filecmp.cmp(
        f"{general_path}data/tfi_data/panel/t-1/panel_trimestral_{nivel}.csv",
        f"{DESTINO_BASE}/sin_estimar/panel_trimestral_{nivel}.csv",
        shallow=False,
    )
    print(f"{nivel}: escenario 'sin_estimar' idéntico al panel_trimestral real -> {igual}")

municipal: escenario 'sin_estimar' idéntico al panel_trimestral real -> True
provincial: escenario 'sin_estimar' idéntico al panel_trimestral real -> True
nacional: escenario 'sin_estimar' idéntico al panel_trimestral real -> True


## 7. Barrido `delta_v` en los 3 escenarios (mismo diseño que `05_5_barrido_k.ipynb`)

Mismo criterio de esa notebook: se mide el tiempo real de una corrida antes de correr la grilla
completa, para decidir `n_alphas` con un número medido, no supuesto.

In [9]:
import time

ORDEN_SUFIJO_K = [
    "_nivel_kt", "_final_kt", "_pendiente_kt", "_volatilidad_kt", "_acum_kt",
    "_nivel_kt1", "_final_kt1", "_pendiente_kt1", "_volatilidad_kt1", "_acum_kt1",
    "_nivel_kd",
]
TARGET = "delta_v"


def corrida_completa(escenario, nivel, modo, k, n_alphas):
    panel_dir = f"{DESTINO_BASE}/{escenario}"
    df = construir_panel_k(nivel, k=k, modo=modo, panel_dir=panel_dir, registro_path=REGISTRO_VARIABLES_PATH)
    cols = columnas_candidatas_k(df, excluir_adicional=(TARGET,))
    corr = df[cols].corr(method="pearson")
    clusters = encontrar_redundantes(corr, 0.90)
    representantes = sorted(elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO_K) for c in clusters)
    paneles = {nivel: df}
    X, y = construir_Xy_final(nivel, representantes, paneles, target=TARGET)
    baseline = baseline_trivial_loocv(y)
    resultado_cv = lasso_loocv_manual(X, y, n_alphas=n_alphas)
    return X, y, baseline, resultado_cv


t0 = time.perf_counter()
X_ref, y_ref, baseline_ref, cv_ref = corrida_completa("con_estimado_desde_t2", "nacional", "delta", 4, n_alphas=10)
t1 = time.perf_counter()
tiempo_por_corrida = t1 - t0
N_ESCENARIOS = 3
N_COMBOS = len(NIVELES) * 2 * 8 * N_ESCENARIOS  # 3 niveles x 2 modos x K=1..8 x 3 escenarios
proyeccion_min = tiempo_por_corrida * N_COMBOS / 60

print(f"Tiempo real de 1 corrida (n_alphas=10): {tiempo_por_corrida:.1f} s")
print(f"Proyección para {N_COMBOS} corridas (3 niveles x 2 modos x 8 K x 3 escenarios): {proyeccion_min:.1f} min")

[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


Tiempo real de 1 corrida (n_alphas=10): 27.1 s
Proyección para 144 corridas (3 niveles x 2 modos x 8 K x 3 escenarios): 65.0 min


In [10]:
filas_barrido = []
for escenario in ESCENARIOS:
    for nivel in NIVELES:
        for modo in ("nivel", "delta"):
            for k in range(1, 9):
                X, y, baseline, cv = corrida_completa(escenario, nivel, modo, k, n_alphas=10)
                idx_min = cv["mean_mse"].argmin()
                mse_min = cv["mean_mse"][idx_min]
                beta = ajustar_final(X, y, cv["alpha_min"])
                activos = beta[beta != 0].sort_values(key=abs, ascending=False)
                filas_barrido.append({
                    "escenario": escenario, "nivel": nivel, "modo": modo, "k": k,
                    "N": X.shape[0], "P": X.shape[1],
                    "alpha_min": cv["alpha_min"],
                    "mse_trivial": baseline, "mse_min": mse_min,
                    "mejora_pct": 100 * (1 - mse_min / baseline) if baseline else None,
                    "n_activos": len(activos),
                    "variables_activas": "; ".join(activos.index) if len(activos) else "(ninguna)",
                })
    print(f"escenario {escenario} completo")

barrido_3_escenarios = pd.DataFrame(filas_barrido)

[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2013_2015']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2013_2015']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


...

[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


escenario con_estimado_desde_t4 completo


In [11]:
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 250)

comparacion = barrido_3_escenarios.pivot_table(
    index=["nivel", "modo", "k"], columns="escenario",
    values=["N", "alpha_min", "mejora_pct", "n_activos"],
)
comparacion

N                                               alpha_min                                              mejora_pct                                               n_activos                                  
escenario          con_estimado_desde_t2 con_estimado_desde_t4 sin_estimar con_estimado_desde_t2 con_estimado_desde_t4 sin_estimar con_estimado_desde_t2 con_estimado_desde_t4 sin_estimar con_estimado_desde_t2 con_estimado_desde_t4 sin_estimar
nivel      modo  k                                                                                                                                                                                                                                
municipal  delta 1                   8.0                   8.0         8.0             10.550745             10.550745   10.550745             -1.726701             -1.726701   -1.726701                   1.0                   1.0         1.0
                 2                   8.0                   8.0         8.0             10.315948             10.315948   10.315948             -4.804239             -6.854976   -6.107321                   0.0                   0.0         0.0
                 3                  10.0                  10.0        10.0             10.179106             10.179106   10.179106              1.183039              1.183039    1.183039                   0.0                   0.0         0.0
                 4                  10.0                  10.0        10.0              9.710364              9.767568    9.710364             -0.071442              0.123602    0.140530                   0.0                   0.0         0.0
                 5                  10.0                  10.0        10.0             10.256941             10.256941   10.256941             -0.289082             -0.289082   -0.289082                   1.0                   1.0         1.0
                 6                  10.0                  10.0        10.0              8.395668              8.395668    8.395668             -4.928155             -5.061620   -5.022395                   0.0                   0.0         0.0
                 7                  10.0                  10.0        10.0              9.362863              9.362863    9.362863             -2.535528             -2.422834   -2.474308                   1.0                   1.0         1.0
                 8                  10.0                  10.0        10.0              5.011156              5.011156    5.011156              0.739625              2.364400    1.712777                   3.0                   3.0         3.0
           nivel 1                  10.0                  10.0        10.0              8.457570              8.457570    8.457570              0.424913              0.424913    0.424913                   0.0                   0.0         0.0
                 2                  10.0                  10.0        10.0              4.731231              8.611393    8.611393              2.907139             -1.407678  -10.799414                   3.0                   0.0         0.0
                 3                  11.0                  11.0        11.0              7.101511              7.101511    7.101511             -4.646868             -4.646868   -4.646868                   0.0                   0.0         0.0
                 4                  11.0                  11.0        11.0              6.578712              6.578712    6.578712             -2.410906             -2.410906   -2.410906                   0.0                   0.0         0.0
                 5                  11.0                  11.0        11.0              6.256154              6.256154    6.256154             -8.757612             -8.757612   -8.757612                   1.0                   1.0         1.0
                 6                  11.0                  11.0        11.0              7.445276              7.445276    7.445276             -2.20836

In [12]:
# Solo las filas (nivel, modo, K) donde el resultado CAMBIA entre escenarios --
# la mayoría de las combinaciones no tocan el trimestre estimado y deberían dar
# exactamente lo mismo en los 3 escenarios (chequeo de consistencia del pipeline).
pivot_mejora = barrido_3_escenarios.pivot_table(index=["nivel", "modo", "k"], columns="escenario", values="mejora_pct")
cambia = pivot_mejora[~np.isclose(pivot_mejora["sin_estimar"], pivot_mejora["con_estimado_desde_t2"], atol=1e-6) |
                       ~np.isclose(pivot_mejora["sin_estimar"], pivot_mejora["con_estimado_desde_t4"], atol=1e-6)]
print(f"{len(cambia)} de {len(pivot_mejora)} combinaciones (nivel, modo, K) cambian entre escenarios")
cambia

33 de 48 combinaciones (nivel, modo, K) cambian entre escenarios


escenario           con_estimado_desde_t2  con_estimado_desde_t4  sin_estimar
nivel      modo  k                                                           
municipal  delta 2              -4.804239              -6.854976    -6.107321
                 4              -0.071442               0.123602     0.140530
                 6              -4.928155              -5.061620    -5.022395
                 7              -2.535528              -2.422834    -2.474308
                 8               0.739625               2.364400     1.712777
           nivel 2               2.907139              -1.407678   -10.799414
                 6              -2.208361              -1.889898    -1.889898
                 7              -1.551028              -1.419544    -1.419544
                 8              12.409299              13.433583    13.433583
nacional   delta 1              29.345249              67.588485    65.809390
                 2              76.975643              78.104385    79.170130
                 3             -11.234750             -10.619599   -10.882290
                 5              -5.400321              -4.922157    -4.922157
                 6             -14.305396              -9.071467    -9.071467
                 7             -19.097695             -17.291360   -17.926020
                 8              -3.310352              -0.293856    -1.497155
           nivel 2             -12.091575             -11.603976   -11.603976
                 5             -15.131697             -13.750797   -13.875498
                 6              -8.538113              -3.612850    -4.177115
                 7             -14.864984             -12.480298   -13.097447
                 8               6.220878              23.931259    23.161014
provincial delta 1              -0.143907              55.310667    60.187176
                 2              54.484837              43.097495    59.928668
                 3             -22.807667             -22.151596   -22.161649
                 4             -16.803377             -16.349587   -16.440405
                 5             -12.205692             -13.773793   -13.803886
                 6              -8.030870              -9.814923   -10.256301
                 7              -7.635264             -10.982237   -11.062166
                 8              -6.309759              -9.620727    -9.835875
           nivel 5             -11.269028             -13.352332   -13.378090
                 6             -19.129831             -28.976610   -28.765751
                 7             -16.657646              22.636864    20.540060
                 8              32.829684              44.282157    44.354059

In [13]:
# variables activas, solo para las combinaciones que sí cambian -- para ver si además de
# la métrica, cambia CUÁL variable queda seleccionada.
if len(cambia):
    claves_cambiantes = set(cambia.index)
    mascara = barrido_3_escenarios.apply(lambda r: (r["nivel"], r["modo"], r["k"]) in claves_cambiantes, axis=1)
    filas_cambiantes = barrido_3_escenarios[mascara].sort_values(["nivel", "modo", "k", "escenario"])
    resultado_cambiantes = filas_cambiantes[["nivel","modo","k","escenario","N","P","alpha_min","mejora_pct","n_activos","variables_activas"]]
else:
    resultado_cambiantes = "ninguna combinación cambió"
resultado_cambiantes

,nivel,modo,k,escenario,N,P,alpha_min,mejora_pct,n_activos,variables_activas
57,municipal,delta,2,con_estimado_desde_t2,8,70,10.315948,-4.804239,0,(ninguna)
105,municipal,delta,2,con_estimado_desde_t4,8,69,10.315948,-6.854976,0,(ninguna)
9,municipal,delta,2,sin_estimar,8,67,10.315948,-6.107321,0,(ninguna)
59,municipal,delta,4,con_estimado_desde_t2,10,69,9.710364,-0.071442,0,(ninguna)
107,municipal,delta,4,con_estimado_desde_t4,10,70,9.767568,0.123602,0,(ninguna)
...,...,...,...,...,...,...,...,...,...,...
118,provincial,nivel,7,con_estimado_desde_t4,11,37,1.745978,22.636864,5,icg_pendiente_kt; tasa_informalidad_volatilidad_kt; reservas_volatilidad_kt; reservas_pendiente_kt; icg_final_kt
22,provincial,nivel,7,sin_estimar,11,37,1.745978,20.540060,5,icg_pendiente_kt; tasa_informalidad_volatilidad_kt; reservas_volatilidad_kt; reservas_pendiente_kt; icg_final_kt
71,provincial,nivel,8,con_estimado_desde_t2,11,39,2.217387,32.829684,4,icg_pendiente_kt; reservas_volatilidad_kt; tasa_informalidad_volatilidad_kt; emae_volatilidad_kt
119,provincial,nivel,8,con_estimado_desde_t4,11,39,2.217387,44.282157,4,icg_pendiente_kt; reservas_volatilidad_kt; tasa_informalidad_volatilidad_kt; reservas_pendiente_kt


## Conclusión

Todo lo de abajo describe únicamente lo que mostraron las celdas de arriba, en esta corrida.

- **Aglomerados de referencia**: Gran Córdoba (13) + Gran Rosario (4), confirmados como
  efectivamente relevados en 3T2007 contra la fuente primaria oficial (`EPHcont_3trim07.pdf`,
  INDEC), únicos dos aglomerados Pampeana de 500.000+ habitantes en esa lista.
- **Limitación de empleo estatal, confirmada con datos reales**: Gran La Plata promedia 32,1% de
  empleo estatal sobre ocupados en los 11 trimestres cargados (rango 28,9%-35,6%), contra 12,9%
  Córdoba y 11,3% Rosario (rangos 9,3%-15,2%) -- 2 a 3 veces más, consistente con la cifra de
  referencia (~38%, IELAP).
- **Validación de paralelismo (2006/2008, 6 pares verificables): el ajuste es pobre para las 6
  variables.** El RMSE de usar el movimiento Córdoba+Rosario como predictor supera el desvío
  estándar del propio movimiento real de Gran La Plata en **5 de 6 variables** (todas menos
  `pct_hogares_vendio_pertenencias`, donde da prácticamente empatado) -- predecir "sin cambio" es
  mejor que usar la variación de referencia, en casi todos los casos. La correlación es baja o
  negativa: `tasa_informalidad` da **-0,56** (el signo opuesto al esperado), consistente con la
  limitación de estructura de empleo. Esta señal es anterior a cualquier estimación de 2007T3 y
  no depende de ella -- es evidencia directa de que el método no sería confiable.
- **Estimación 2007T3, guardada en `data/tfi_data/eph_estimacion_2007t3_experimental.csv`** (6
  filas, `estimado_no_medido=True`), con `estimado_desde_t2`/`estimado_desde_t4` calculados por
  separado (nunca promediados, por la cancelación algebraica ya explicada) -- la divergencia entre
  ambas proyecciones va de 0,011 (`pct_hogares_ayuda_social_gobierno`) a 0,085
  (`tasa_informalidad`), grande en relación a la variabilidad trimestral típica de cada variable.
- **Los 3 escenarios de `panel_trimestral`**: confirmado que `sin_estimar` es **byte-idéntico** al
  panel real (`filecmp.cmp` `True` en los 3 niveles) -- el experimento no contamina nada existente.
- **Barrido**: 1 corrida real midió 27,1 s con `n_alphas=10`; la proyección de 144 corridas
  (3 niveles x 2 modos x 8 K x 3 escenarios) a ese mismo `n_alphas=10` dio 65,0 min reales --
  corrida completa dentro del mismo notebook, sin script aparte, igual criterio que
  `05_5_barrido_k.ipynb`.
- **Resultado de sensibilidad: 33 de 48 combinaciones (nivel, modo, K) cambian de resultado según
  el escenario** -- más de dos tercios de la grilla. Dos casos ilustran la magnitud real del
  efecto:
  - `provincial, delta, K=1`: `mejora_pct` pasa de **-0,1%** (`con_estimado_desde_t2`) a **+55,3%**
    (`con_estimado_desde_t4`) a **+60,2%** (`sin_estimar`) -- la sola elección de qué ancla (T2 o
    T4) usar para la estimación cambia la conclusión de "sin señal" a "mejora fuerte".
  - `nacional, delta, K=1`: 29,3% vs. 67,6% vs. 65,8% -- mismo patrón, magnitud menor.
  - En varias filas (ej. `municipal, nivel, K=2`) el signo de `mejora_pct` se invierte entre
    escenarios (+2,9% vs. -1,4% vs. -10,8%).
- **Lectura del conjunto**: la combinación de (a) una validación que ya muestra ajuste pobre en
  los períodos verificables y (b) una sensibilidad de más de dos tercios de la grilla `delta_v` a
  la sola elección de ancla, es la misma señal en dos lugares distintos -- este método de
  estimación no debería tratarse como un dato confiable para ningún análisis, aunque sí sirve para
  lo que este notebook se propuso: medir cuánto puede llegar a cambiar un resultado por un solo
  trimestre estimado con un método razonable pero no validado. Ninguna fila de este notebook
  reemplaza el `NaN` real en ningún archivo de producción.
